# Make a short (45 second) AI song on the Qualcomm IQ8 NPU

**What you get:** a **44.6-second** song generated from a style description and lyrics. The heavy part of the AI model runs on the **NPU** (the neural chip) of a Qualcomm IQ8 board; the rest runs on your Windows PC.

**Why 45 seconds:** DiffRhythm's stock song is 95 seconds. Half the length means half the work per step, so a song takes about 4 to 6 minutes instead of 10 to 13.

**Who this is for:** makers and developers with an IQ8 and a Windows PC who want a quicker song and are happy with 45 seconds.

## The short version

| Question | Answer |
| --- | --- |
| What do I run to make a song? | If the graphs are already built (the one-time build below): **Section 1, Section 2, Section 12.** Nothing else. |
| How long does a song take? | **About 4 to 6 minutes** on a quiet PC. The NPU part is a steady 81 s; the rest is PC work and varies. See Results. |
| What is the one-time build? | The NPU graphs are compiled for one fixed song length, so a 45 s song needs its own: export, convert, quantize and compile at 960 frames (Sections 3 to 11). **A few hours, mostly waiting, once.** Finished steps are skipped on a re-run. |
| What does it need? | An IQ8 on your network, and a Windows PC with WSL, the DiffRhythm install in WSL and the QAIRT SDK (see `DIFFRHYTHM_IQ8.md`). |
| Is it accurate? | Yes. The accuracy gate scores 0.99986 against the full-precision reference (0.99 is required; 0.999 or more is "as good as fp16"). |
| Does it sound good? | The gate checks the numbers, not the music. DiffRhythm was trained on 95 s songs, so 45 s is outside what it saw. Listen to your song; the checks here only rule out silence, noise and clipping. |
| Will it break my other setups? | No. Everything lives in separate folders (`seg_45s`, `ctx_45s`, `ctx_45s_w8a16`). |

## Results (measured on the IQ8, HTP v75, 16 sampler steps)

**Speed.** NPU time is steady; the total is not, because the PC part (text and style encoders, audio decoder) varies with how busy the PC and the network are.

| | 95 s song, mixed int8 | **45 s song, mixed int8** | 45 s song, all fp16 |
| --- | --- | --- | --- |
| Song length | 95.1 s | **44.6 s** (960 frames) | 44.6 s |
| NPU time per transformer pass | 6.2 s | **2.5 s** | 3.7 s |
| NPU time per song (30 passes), every run | 194 s (3 runs: 194.4 to 194.7) | **81 s** (4 runs: 80.4 to 81.5) | 113 s |
| Song cell, wall clock, quiet PC | 8 to 12 min | **3.7 to 6 min** | 4.9 to 5.3 min |
| Song cell, wall clock, repeat runs | 9.8, 14.4, 12.8 min (mean 12.3) | **6.0, 5.9, 5.5, 12.1 min** (median 5.9) | not repeated |
| Gate cosine vs fp32 reference | 0.99998 | **0.99986** | 0.99999 |

How to read it: in the first runs a 45 s song took 3.7 to 4.2 minutes; in the repeat runs, with the board reached over a different network path and a busier PC, three of four runs took 5.5 to 6.0 minutes and one took 12.1 minutes. In every run the NPU part was the same (81 s), so the whole spread is PC-side work. **Typical for you: plan on about 6 minutes per 45 s song, and expect it to be 4 minutes on a quiet PC.** The 95 s song took about 2 times as long in the same runs, so the saving is real but depends on the PC: about 40 to 60% less time, and per second of music the two are about the same (0 to 15% better for the short song), because the shorter song is mostly just less music.

**Board stability (the IQ8 can reset while loading its model graphs).**

| Setup | Loads | Board resets |
| --- | --- | --- |
| 45 s mixed int8 | 6 | 0 |
| 95 s mixed int8 | 6 | 0 |
| 45 s all fp16 | 6 | 0 |
| 95 s all fp16 (the stock graphs, 2.3 GB) | 2 | **2** (each took the board offline for 10 minutes or more, the second with 3 `external abort` messages) |

Only the stock 95 s fp16 load reset the board in this test. Six clean loads only prove the reset rate is below about 1 in 2 per load, not that it is zero, and the test cannot say whether int8 or the shorter length is the reason. Treat it as encouraging, not as proof.

## Pick your path

| I want to... | Run these sections | Time |
| --- | --- | --- |
| **Make 45 s songs (after the one-time build)** | 1 Configuration, then the **Song** cell (section 12) | see Results |
| **Do the one-time build** | 1 to 11 in order, then 12 | a few hours |
| **Compare against the stock-precision 45 s song** | 13 (optional) | one more song |

## Words used in this notebook

| Word | Definition |
| --- | --- |
| **NPU** | The neural processing chip on the board. The heavy part of the model runs here. |
| **Frames** | The model works on slices of audio, 21.5 per second. 960 frames is 44.6 s; 2048 frames is the stock 95 s. |
| **fp16 / int8** | 16-bit floats (precise, bigger) and 8-bit integers (smaller, faster, less precise). |
| **Mixed precision** | Four chunks of the model: the first and last stay fp16, the two middle ones go int8. This is the setup that keeps the sound intact. |
| **Calibration** | Showing the converter real examples so int8 numbers get sensible ranges. |
| **Gate** | A pass/fail check run before making a song: the cosine score (1.0 = identical to the reference) must be at least 0.99. |
| **Context binary** | The model compiled for the NPU as a file on the board. |

The reasoning behind each step is in `DIFFRHYTHM_IQ8.md`; this notebook explains the steps as it runs them.

## What is different from the stock 95 s song

1. The model is exported with `DR_FRAMES=960` instead of 2048 (`export_segments.py`), into its own folder.
2. `infer_iq8.py` cuts DiffRhythm's lyric tokens and empty audio prompt to 960 frames, so the sampler, the NPU passes and the audio decoder all work on 44.6 s.
3. The all-int8 graphs are skipped: they fail the quality gate on this model (cosine 0.61 at 95 s), so only the mixed setup is built.
4. Lyrics must fit inside 44 s. **DiffRhythm treats the last lyric line as an end marker and does not sing it**, so add one extra line at the end.

## Before you start

- Three environment variables, set in the terminal you launch VS Code or Jupyter from (start VS Code with `code .` from that same terminal, with no other VS Code window open): `IQ8_HOST` (board IP), `IQ8_HOSTKEY` (its SSH host-key fingerprint, `SHA256:...`) and `QLI_SSH_PASSWORD`. The notebook never stores them.
- A 64-bit x64 Python 3.10 on the PC for the quantizer, and about 20 GB free on the work disk.
- A speaker or headphones on the board's audio jack, if you want to hear the song on the board.

## 1. Configuration

Defines the settings and helper functions, then sets the short-song options. Run it once per session; later cells depend on it.

In [ ]:
# Settings and helper functions (wsl, ssh, push, pull, wait_board). Board details come from env vars, never stored.
# Settings. Edit STEPS / STYLE_PROMPT / LYRICS_LRC to change the song; board details come from env vars (never stored).
import os, re, shutil, struct, math, subprocess, time, json, wave, array, tarfile, hashlib, shlex
from pathlib import Path

QAIRT = Path(os.environ.get("QAIRT_ROOT", r"C:\Qualcomm\AIStack\QAIRT\2.51.40.260930"))
WORK  = Path(os.environ.get("DR_IQ8_WORK", r"C:\dr_iq8"))          # staging on a plain local disk (NOT Dropbox)
OUT   = Path("diffrhythm_outputs"); OUT.mkdir(exist_ok=True)
SCRIPTS = Path("diffrhythm_npu_scripts").resolve()                   # export_segments.py, dit_server.cpp, infer_iq8.py

IQ8_HOST    = os.environ["IQ8_HOST"]
IQ8_HOSTKEY = os.environ["IQ8_HOSTKEY"]
IQ8_PW      = os.environ["QLI_SSH_PASSWORD"]
BOARD = "/root/diffrhythm"                                           # working dir on the board
SEGS  = ["seg0", "seg1", "seg2", "seg3"]

WSL_DISTRO = os.environ.get("DR_WSL_DISTRO", "Ubuntu-22.04")
WSL_USER   = os.environ.get("DR_WSL_USER", "ubuntu")
WSL_ROOT   = os.environ.get("DR_WSL_ROOT", "/home/ubuntu/diffrhythm_xelite")

STEPS = int(os.environ.get("DR_STEPS", "16"))      # Euler steps; the DiT runs 2*(STEPS-1) times (CFG). 16 = about 11 min per song; 32 = original quality, about 20 min.
STYLE_PROMPT = "lofi, piano, rain, calm, instrumental."
# No empty-text lines: DiffRhythm's g2p raises "Unknown language" on an empty lyric line.
LYRICS_LRC = """[00:10.00]Quiet lights across the bay
[00:16.00]Edge devices dream all day
[00:22.00]Little chips that learn to sing
[00:28.00]Music made from everything
"""
WORK.mkdir(parents=True, exist_ok=True)
SEGDIR = WORK / "seg"; SEGDIR.mkdir(exist_ok=True)

def to_wsl_path(p):
    p = Path(p).resolve(); return "/mnt/" + p.drive.rstrip(":").lower() + p.as_posix()[2:]
print("board", IQ8_HOST, "| work", WORK, "| wsl root", WSL_ROOT)

# Helpers: wsl() runs bash in WSL, ssh()/push()/pull()/wait_board() talk to the board with plink/pscp (always -ssh).
def wsl(script, timeout=3600, check=True):
    cmd = ["wsl", "-d", WSL_DISTRO, "-u", WSL_USER, "--", "bash", "-s"]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    p.stdin.write(("set -o pipefail\n" + script).replace("\r\n", "\n").encode()); p.stdin.close()
    lines, t0 = [], time.time()
    for raw in p.stdout:
        line = raw.decode("utf-8", "replace").replace("\r", ""); lines.append(line)
        print(line.encode("ascii", "backslashreplace").decode(), end="")
        if time.time() - t0 > timeout: p.kill(); raise TimeoutError(f"WSL step exceeded {timeout}s")
    rc = p.wait()
    if check and rc: raise RuntimeError(f"WSL step failed rc={rc}")
    return rc, "".join(lines)

PL = ["-batch", "-hostkey", IQ8_HOSTKEY, "-pw", IQ8_PW]
def ssh(cmd, timeout=120, retries=3, quiet=False):
    last = ""
    for i in range(retries):
        try:
            p = subprocess.run(["plink", "-ssh", *PL, f"root@{IQ8_HOST}", cmd], capture_output=True, text=True, timeout=timeout, stdin=subprocess.DEVNULL)
            both = (p.stdout or "") + (p.stderr or "")
            if "plink:" in both and p.returncode != 0:      # e.g. "the -pw option can only be used with the SSH protocol"
                raise RuntimeError("plink failed: " + both.strip()[:300])
            if "FATAL ERROR" in both or "Access denied" in both:   # plink prints these on stdout or stderr
                if "Access denied" in both or "password was not accepted" in both:
                    raise RuntimeError("SSH login refused: QLI_SSH_PASSWORD in this session is wrong for the board. Set it in the terminal you launch VS Code from, then restart the kernel.")
                last = both
            else:
                out = (p.stdout or "").strip()
                if not quiet and out: print(out)
                return p.returncode, out
        except subprocess.TimeoutExpired: last = "timeout"
        time.sleep(5)
    raise RuntimeError(f"ssh failed after {retries} tries: {last.strip()[:200]}")

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()

def push(local, remote_dir, chunk=16 << 20):
    local = Path(local); name = local.name; want = md5(local)
    # Big files: compare sizes only, so a re-run does not make the board read ~6 GB just to say "up to date"
    # (this board has a memory fault that heavy memory traffic can trigger). Small files: full md5.
    if local.stat().st_size > (64 << 20):
        _, got = ssh(f"mkdir -p {remote_dir}; stat -c %s {remote_dir}/{name} 2>/dev/null", quiet=True)
        if got.strip() == str(local.stat().st_size): print("up to date (same size):", name); return
    else:
        _, got = ssh(f"mkdir -p {remote_dir}; md5sum {remote_dir}/{name} 2>/dev/null | cut -d' ' -f1", quiet=True)
        if got.strip() == want: print("up to date:", name); return
    parts = WORK / "chunks" / name; shutil.rmtree(parts, ignore_errors=True); parts.mkdir(parents=True)
    with open(local, "rb") as f:
        for i, data in enumerate(iter(lambda: f.read(chunk), b"")): (parts / f"p.{i:04d}").write_bytes(data)
    rp = f"{remote_dir}/.parts_{name}"
    def ensure_dir():
        p = subprocess.run(["plink", "-ssh", *PL, f"root@{IQ8_HOST}", f"mkdir -p {rp} && test -d {rp} && echo yes || echo no"],
                           capture_output=True, text=True, timeout=60, stdin=subprocess.DEVNULL)
        if p.stdout.strip() != "yes":
            raise RuntimeError(f"cannot create {rp} on the board: rc={p.returncode} stdout={p.stdout!r} stderr={p.stderr!r} plink={shutil.which('plink')}")
    ensure_dir()
    for c in sorted(parts.iterdir()):
        w = md5(c)
        err = ""
        for _ in range(6):
            _, g = ssh(f"md5sum {rp}/{c.name} 2>/dev/null | cut -d' ' -f1", quiet=True)
            if g.strip() == w: break
            ensure_dir()
            r = subprocess.run(["pscp", *PL, "-q", str(c), f"root@{IQ8_HOST}:{rp}/{c.name}"], capture_output=True, text=True, timeout=600, stdin=subprocess.DEVNULL)
            err = ((r.stdout or "") + (r.stderr or "")).strip()
        else: raise RuntimeError(f"chunk {c.name} failed. pscp said: {err[:400] or '(nothing)'} | pscp = {shutil.which('pscp')}")
    ssh(f"cd {remote_dir} && cat {rp}/p.* > {name} && rm -rf {rp}", timeout=300)
    _, g = ssh(f"md5sum {remote_dir}/{name} | cut -d' ' -f1", quiet=True)
    assert g.strip() == want, f"md5 mismatch for {name}"
    shutil.rmtree(parts, ignore_errors=True); print("pushed", name)

def wait_board(max_s=900):
    """Block until the board answers SSH again (it resets now and then; see DIFFRHYTHM_IQ8.md)."""
    t0 = time.time()
    while time.time() - t0 < max_s:
        try:
            ssh("true", retries=1, timeout=20, quiet=True); return True
        except Exception as e:
            if "SSH login refused" in str(e) or "plink failed" in str(e): raise
            time.sleep(15)
    return False

def pull(remote, local):
    subprocess.run(["pscp", *PL, "-q", f"root@{IQ8_HOST}:{remote}", str(local)], check=True, capture_output=True, timeout=600, stdin=subprocess.DEVNULL)

ENV = f"""
export R={WSL_ROOT}
export PY=$R/env/bin/python
export PYTHONNOUSERSITE=1 HF_HOME=$R/hf_home
export SDK={to_wsl_path(QAIRT)}
export S={to_wsl_path(SCRIPTS)}
"""

# ---- short-song settings ----
FRAMES = int(os.environ.get("DR_FRAMES", "960"))           # 21.5 frames per second: 960 = 44.6 s (2048 = the stock 95 s song)
SONG_SECONDS = FRAMES * 2048 / 44100
STYLE_PROMPT = "acoustic folk, guitar, warm, gentle, female vocal."
LYRICS_45 = """[00:04.00]Quiet lights across the bay
[00:10.00]Edge devices dream all day
[00:16.00]Little chips that learn to sing
[00:22.00]Music made from everything
[00:30.00]Morning comes and the night is done
[00:36.00]The last line is only an end marker
"""   # keep every line under 44 s; DiffRhythm does not sing the LAST line (it marks the end), and empty lines make the text step fail
assert not any(not l.strip() for l in LYRICS_45.strip().splitlines()), "remove empty lines from the lyrics"

# Where everything lives (all separate from the stock 95 s setup, which is never touched)
SEGD, CASE = "seg_45s", "case_45s"                         # WSL folder (under $R/src) and board folders: fp16 DLCs and the test case
CTX16 = "ctx_45s"                                          # board folder: fp16 context binaries
SEGQ, CTXQ = "seg_45s_w8a16", "ctx_45s_w8a16"              # board folders: int8 DLCs and context binaries
SEG45 = WORK / "seg_45s"; SEG45.mkdir(exist_ok=True)       # PC copies of the DLCs and the test case
Q45 = WORK / "q45_w8a16"; Q45.mkdir(exist_ok=True)         # calibration lists, fp32 DLCs, quantized DLCs
MIX = f"{CTX16},{CTXQ},{CTXQ},{CTX16}"                     # mixed precision: segments 0 and 3 fp16, 1 and 2 int8
CALIB_STEPS   = 8                                          # sampler steps per calibration run (2*(8-1) = 14 passes each)
CALIB_PROMPTS = [                                          # more variety = better ranges; each run costs a few minutes
    ("lofi, piano, rain, calm, instrumental.", LYRICS_45),
    ("upbeat electronic pop, synth, drums, energetic.", "[00:04.00]Lights are rising in the night\n[00:10.00]Every circuit shining bright\n[00:16.00]Dance with me until the dawn\n[00:22.00]Silicon and song go on\n[00:30.00]End marker line\n"),
]
N_CALIB = 8
SCHEME, W_BITS, A_BITS = "w8a16", 8, 16                    # weights int8, activations 16-bit (the only scheme that works on this model)
QPY = Path(os.environ.get("DR_QUANT_PY", str(Path(os.environ["LOCALAPPDATA"]) / "Programs/Python/Python310/python.exe")))   # x64 Python 3.10
print(f"board {IQ8_HOST} | {FRAMES} frames = {SONG_SECONDS:.1f} s | steps {STEPS} | work {WORK} | quantizer python {QPY}")

## 2. Preflight (read-only)

Checks the tools on the PC, the DiffRhythm install in WSL, and the board. Nothing is changed. Stop if anything shows `MISSING`: the WSL or QAIRT setup is missing, or the board is unreachable.

In [ ]:
SDK_WIN = QAIRT
for n in [QPY, SDK_WIN / "bin/x86_64-windows-msvc/qairt-quantizer", SDK_WIN / "lib/x86_64-windows-msvc",
          SCRIPTS / "export_segments.py", SCRIPTS / "make_calib.py", SCRIPTS / "gate_board.py", SCRIPTS / "infer_iq8.py", SCRIPTS / "dit_server.cpp"]:
    print("ok     " if n.exists() else "MISSING", n)
wsl(ENV + r"""
for f in $PY $R/src/infer/infer.py $R/src/pretrained $HOME/miniforge3/envs/qairt312/bin/python; do [ -e $f ] && echo "ok      $f" || echo "MISSING $f"; done
$R/env/bin/python -c "import onnxruntime; print('ok      onnxruntime', onnxruntime.__version__)" 2>&1 | tail -1
df -h $R | tail -1
""")
import shutil as _sh
print("PC free disk on the work drive: %.0f GB" % (_sh.disk_usage(WORK).free / 2**30))
_ = ssh("uptime; echo aborts_since_boot=$(dmesg | grep -c 'external abort'); ls %s/dit_server %s/qnn_rt/bin/qnn-context-binary-generator %s/cfg_seg0.json 2>&1; df -h %s | tail -1" % ((BOARD,) * 4))

## 3. Export the short ONNX segments (WSL, a few minutes)

Cuts the DiffRhythm transformer into the same four chunks as before, but with a sequence of `FRAMES` (960) audio frames instead of 2048. The script also checks that the four chunks chained together give the same answer as the full PyTorch model (it must, before it exports anything), and writes a small fixed test case with the expected output, used later as the accuracy gate.

In [ ]:
_ = wsl(ENV + f"""
set -e; cd $R/src
if [ -f {SEGD}/seg3.onnx ] && [ -f {SEGD}/case/ref_v.f32 ]; then echo "segments already exported"; else
  cp $S/export_segments.py $R/export_segments.py
  DR_FRAMES={FRAMES} DR_SEGDIR={SEGD} HF_HUB_OFFLINE=1 $PY $R/export_segments.py 2>&1 | grep -v Warning | tail -12
fi
ls -la {SEGD}/*.onnx
""", timeout=3600)

## 4. Convert to fp16 DLCs (WSL, about 1 minute each)

A DLC is Qualcomm's model container. These fp16 versions become the first and last chunks of the mixed setup, and also the source for calibration data.

In [ ]:
_ = wsl(ENV + f"""
set -e; cd $R/src
ENV312=$HOME/miniforge3/envs/qairt312
export PYTHONPATH=$SDK/lib/python LD_LIBRARY_PATH=$SDK/lib/aarch64-oe-linux-gcc11.2:${{LD_LIBRARY_PATH:-}}
for i in 0 1 2 3; do
  if [ -f {SEGD}/seg$i.dlc ]; then echo "seg$i.dlc exists"; continue; fi
  echo "== seg$i"; $ENV312/bin/python $SDK/bin/x86_64-linux-clang/qairt-converter --input_network {SEGD}/seg$i.onnx --float_bitwidth 16 --output_path {SEGD}/seg$i.dlc > {SEGD}/seg$i.convert.log 2>&1
done
ls -la {SEGD}/*.dlc
""", timeout=3600)

## 5. Board runtime and fp16 context binaries

Copies the DLCs and the test case to the board (checked chunk by chunk), makes sure the board has the current `dit_server` (the version that converts between fp16 and fp32 at chunk boundaries, needed for the mixed setup), and compiles each chunk into a **context binary** on the board. Compiling is a heavy job for the board; the known memory fault can reset it, and the loop waits and retries.

In [ ]:
# 1) the board's dit_server must be the current one (with the fp16<->fp32 boundary cast). Rebuild in WSL and push (skipped if identical).
wsl(ENV + f"bash $S/build_dit_server.sh && cp $S/dit_server $R/dit_server && cp $R/dit_server {to_wsl_path(WORK)}/dit_server")
_, stale = ssh("pgrep -c dit_server || true", quiet=True)
assert stale.strip() in ("", "0"), "dit_server is running on the board; reboot it first"
push(WORK / "dit_server", BOARD)
ssh(f"chmod +x {BOARD}/dit_server")

# 2) DLCs and test case: WSL -> PC work disk -> board
wsl(ENV + f"""
cp $R/src/{SEGD}/seg?.dlc {to_wsl_path(SEG45)}/
mkdir -p {to_wsl_path(SEG45)}/case && cp $R/src/{SEGD}/case/* {to_wsl_path(SEG45)}/case/
""")
for n in SEGS: push(SEG45 / f"{n}.dlc", f"{BOARD}/{SEGD}")
for f in sorted((SEG45 / "case").iterdir()): push(f, f"{BOARD}/{CASE}")

In [ ]:
# 3) HTP config files (same as the stock notebook; written only if missing) and the generic context-binary build script
GENQ = """#!/bin/sh
# usage: gen_q.sh segN seg_dir ctx_dir   (writes <ctx_dir>/segN.bin, then <ctx_dir>/segN.ok on success)
cd /root/diffrhythm; mkdir -p $3
R=/root/diffrhythm/qnn_rt
export LD_LIBRARY_PATH=$R/lib ADSP_LIBRARY_PATH="$R/lib/hexagon-v75/unsigned;/usr/lib/rfsa/adsp;/dsp"
n=$1; rm -f $3/$n.ok $3/$n.bin
$R/bin/qnn-context-binary-generator --backend $R/lib/libQnnHtp.so --model $R/lib/libQnnModelDlc.so --dlc_path $2/$n.dlc \\
  --binary_file $n --output_dir $3 --config_file cfg_$n.json --log_level warn > genq_$n.log 2>&1
rc=$?; echo "exit=$rc" >> genq_$n.log
[ $rc -eq 0 ] && [ -s $3/$n.bin ] && touch $3/$n.ok
"""
setup = "cd %s && cat > gen_q.sh <<'EOF'\n%sEOF\nchmod +x gen_q.sh\n" % (BOARD, GENQ)
for n in SEGS:
    setup += ("[ -f cfg_%s.json ] || echo '{\"backend_extensions\":{\"shared_library_path\":\"%s/qnn_rt/lib/libQnnHtpNetRunExtensions.so\",\"config_file_path\":\"%s/htp_%s.json\"}}' > cfg_%s.json\n" % (n, BOARD, BOARD, n, n))
    setup += ("[ -f htp_%s.json ] || echo '{\"graphs\":[{\"graph_names\":[\"%s\"],\"vtcm_mb\":8,\"fp16_relaxed_precision\":1,\"O\":3}],\"devices\":[{\"htp_arch\":75}]}' > htp_%s.json\n" % (n, n, n))
ssh(setup, quiet=True)

def build_contexts(seg_dir, ctx_dir):
    """Compile seg0..3 from <seg_dir>/*.dlc into <ctx_dir>/*.bin on the board, detached, with polling and retry after a board reset."""
    for n in SEGS:
        for attempt in range(1, 4):
            _, ok = ssh(f"[ -f {BOARD}/{ctx_dir}/{n}.ok ] && echo yes || echo no", quiet=True)
            if ok == "yes": print(n, "context binary ready"); break
            print(f"{n}: generating (attempt {attempt})")
            # Launch once: a timed-out launch must NOT be retried blindly, because every launch restarts the job.
            _, busy = ssh(f"pgrep -f 'qnn-context-binary-generato[r].*{n}' >/dev/null && echo run || echo no", quiet=True)
            if busy == "no":
                try: ssh(f"cd {BOARD} && setsid nohup ./gen_q.sh {n} {seg_dir} {ctx_dir} > /dev/null 2>&1 < /dev/null &", quiet=True, retries=1, timeout=90)
                except Exception as e: print("  launch command did not return (board busy):", str(e)[:80], "- checking whether the job started")
            t0 = time.time()
            while time.time() - t0 < 1500:
                time.sleep(20)
                try:
                    _, st = ssh(f"[ -f {BOARD}/{ctx_dir}/{n}.ok ] && echo done || (pgrep -f 'qnn-context-binary-generato[r].*{n}' >/dev/null && echo run || echo dead)", retries=1, timeout=30, quiet=True)
                except Exception: st = "unreachable"
                if st in ("done", "dead"): break
                if st == "unreachable": print("  board unreachable, waiting..."); wait_board()
            print(f"  {n}: {st} after {time.time()-t0:.0f}s")
        else:
            _, lg = ssh(f"tail -15 {BOARD}/genq_{n}.log", quiet=True); print(lg)
            raise RuntimeError(f"{n} failed three times; see {BOARD}/genq_{n}.log")
    _ = ssh(f"ls -la {BOARD}/{ctx_dir} | grep bin; echo aborts_since_boot=$(dmesg | grep -c 'external abort')")

build_contexts(SEGD, CTX16)

## 6. Accuracy gate for the fp16 short graphs

Runs the fixed test case through the four fp16 context binaries on the NPU and compares the result with the PyTorch reference computed in the export step. **This is the first proof that the 960-frame graphs are right.** The cosine should be about 1.0; below 0.99 stop, because a song made from broken graphs is noise. The pass time printed here is the number to compare with the 95 s graphs (9.0 s per pass in fp16).

In [ ]:
def run_gate(ctx, tag, case):
    """Run gate_board.py on the board for context dir(s) `ctx` with test-case dir `case`; return (cosine, max_abs_err, gate_output_text)."""
    push(SCRIPTS / "gate_board.py", BOARD)
    for attempt in range(1, 5):
        assert wait_board(), "board did not come back"
        _, stale = ssh("pgrep -c dit_server || true", quiet=True)
        assert stale.strip() in ("", "0"), "stale dit_server on the board (NPU may be wedged): reboot the board, then re-run"
        ssh(f"cd {BOARD} && rm -f {case}/v_{tag}.raw gate_{tag}.out && setsid nohup timeout 200 python3 gate_board.py {ctx} {case}/v_{tag}.raw {case} > gate_{tag}.out 2>&1 < /dev/null &", quiet=True)
        t0, status = time.time(), "running"
        while time.time() - t0 < 300:
            time.sleep(10)
            try:
                _, status = ssh(f"cd {BOARD} && if [ -s {case}/v_{tag}.raw ]; then echo done; elif pgrep -f 'gate_boar[d].py' >/dev/null; then echo run; else echo dead; fi", retries=1, timeout=30, quiet=True)
            except Exception as e:
                if "SSH login refused" in str(e) or "plink failed" in str(e): raise
                status = "unreachable"
            if status in ("done", "dead", "unreachable"): break
        if status == "done": break
        out = ""
        try: _, out = ssh(f"cat {BOARD}/gate_{tag}.out; tail -5 {BOARD}/server.err", quiet=True)
        except Exception: pass
        print(f"gate attempt {attempt}: {status} after {time.time()-t0:.0f}s {out[-300:]}; waiting for the board and trying again")
        wait_board(); time.sleep(60)
    else:
        raise RuntimeError("gate failed 4 times (board resets or server error); see server.err on the board")
    _, out = ssh(f"cat {BOARD}/gate_{tag}.out", quiet=True)
    pull(f"{BOARD}/{case}/v_{tag}.raw", WORK / f"v_{tag}.raw")
    raw = (WORK / f"v_{tag}.raw").read_bytes()
    a = struct.unpack(("<%df" % (len(raw) // 4)) if len(raw) == FRAMES * 64 * 4 else ("<%de" % (len(raw) // 2)), raw)   # fp32 or fp16 output
    rb = (SEG45 / "case" / "ref_v.f32").read_bytes(); b = struct.unpack("<%df" % (len(rb) // 4), rb)
    assert len(a) == len(b), f"output has {len(a)} values, reference {len(b)}: graphs and test case disagree on the song length"
    cos = sum(x * y for x, y in zip(a, b)) / math.sqrt(sum(x * x for x in a) * sum(y * y for y in b))
    return cos, max(abs(x - y) for x, y in zip(a, b)), out

cos16, err16, gout = run_gate(CTX16, "fp16", CASE)
print(gout)
print(f"fp16 short graphs vs fp32 PyTorch: cosine {cos16:.5f}, max abs err {err16:.3f}")
assert cos16 > 0.99, "the 960-frame fp16 graphs do not match the reference; do not continue"

## 7. Capture calibration data (a few minutes per prompt)

int8 needs real examples to learn how big the numbers get. This runs the **fp16 short graphs** through DiffRhythm once per calibration prompt and saves every transformer input (both the conditioned and the unconditioned pass, spread over the whole denoising trajectory). The generated songs are thrown away. The NPU is busy, so nothing else may use the board meanwhile. A prompt whose dump already exists is skipped.

In [ ]:
DUMP = f"{WSL_ROOT}/calib_dump_45s"
for k, (style, lrc) in enumerate(CALIB_PROMPTS):
    d = f"{DUMP}/prompt{k}"
    _, have = wsl(f"ls {d}/p*_x.raw 2>/dev/null | wc -l", check=False)
    if int(have.strip().splitlines()[-1] or 0) >= 2 * (CALIB_STEPS - 1):
        print(f"prompt{k}: dump already complete"); continue
    print(f"prompt{k}: {style}")
    run = f"{WSL_ROOT}/runs/calib45_{k}"
    wsl(ENV + f"rm -rf {d}; mkdir -p {run} {d}; cat > {run}/lyrics.lrc <<'EOF'\n{lrc}EOF\ncp $S/infer_iq8.py $R/infer_iq8.py\n")
    wsl(ENV + f"""
export PHONEMIZER_ESPEAK_LIBRARY=$($PY -c "import espeakng_loader as e;print(e.get_library_path())")
export ESPEAK_DATA_PATH=$($PY -c "import espeakng_loader as e;print(e.get_data_path())")
export OMP_NUM_THREADS=$(nproc)
export DR_STEPS={CALIB_STEPS} DR_FRAMES={FRAMES} DR_CTX_DIR={CTX16} DR_CALIB_DIR={d} IQ8_HOST={shlex.quote(IQ8_HOST)} IQ8_HOSTKEY={shlex.quote(IQ8_HOSTKEY)} IQ8_PW={shlex.quote(IQ8_PW)}
cd $R/src
$PY $R/infer_iq8.py --lrc-path {run}/lyrics.lrc --ref-prompt "{style}" --audio-length 95 --output-dir {run} --chunked 2>&1 | grep -E "iq8[]] (short|sampler|total|pass failed)|Error|Traceback" | tail -20
""", timeout=3 * 3600)
_ = wsl(f"for d in {DUMP}/*; do echo $d $(ls $d/p*_x.raw | wc -l) passes; done")

## 8. Build the calibration lists (WSL, CPU, a few minutes)

Picks `N_CALIB` passes evenly across everything captured and pushes each one through the fp32 ONNX segments on the CPU, so segments 1 to 3 get the exact tensors the previous segment would produce. Writes the lists the quantizer reads.

In [ ]:
CAL = Q45 / "calib"
if all((CAL / f"{s}_list_win.txt").exists() for s in SEGS):
    print("calibration lists already built:", CAL)
else:
    shutil.rmtree(CAL, ignore_errors=True); CAL.mkdir(parents=True)
    wsl(ENV + f"cd $R/src && $R/env/bin/python $S/make_calib.py {DUMP} $R/src/{SEGD} {to_wsl_path(CAL)} {N_CALIB} 2>&1 | grep -vi warn | tail -25", timeout=3600)
    for s in SEGS:
        txt = (CAL / f"{s}_list.txt").read_text()
        txt = re.sub(r"/mnt/([a-z])/(\S+)", lambda m: m.group(1).upper() + ":\\" + m.group(2).replace("/", "\\"), txt)   # /mnt/c/x -> C:\x for the Windows quantizer
        (CAL / f"{s}_list_win.txt").write_text(txt)
print((CAL / "seg0_list_win.txt").read_text().splitlines()[0][:300])

## 9. Convert to fp32 DLCs (WSL, about 1 minute each)

The quantizer starts from a float model, so these are converted without the fp16 flag. Each is about 1 GB.

In [ ]:
FP32 = Q45 / "fp32"; FP32.mkdir(exist_ok=True)
_ = wsl(ENV + f"""
set -e; cd $R/src
ENV312=$HOME/miniforge3/envs/qairt312
export PYTHONPATH=$SDK/lib/python LD_LIBRARY_PATH=$SDK/lib/aarch64-oe-linux-gcc11.2:${{LD_LIBRARY_PATH:-}}
for i in 0 1 2 3; do
  out={to_wsl_path(FP32)}/seg$i.dlc
  if [ -s $out ]; then echo "seg$i fp32 dlc exists"; continue; fi
  echo "== seg$i"; $ENV312/bin/python $SDK/bin/x86_64-linux-clang/qairt-converter --input_network {SEGD}/seg$i.onnx --output_path $out > {SEGD}/seg$i.fp32.convert.log 2>&1
done
ls -la {to_wsl_path(FP32)}/*.dlc
""", timeout=3600)

## 10. Quantize to int8 (Windows x64 Python, about 1 to 5 minutes per segment)

`qairt-quantizer` runs the calibration samples through each fp32 DLC and writes a DLC with int8 weights (activations 16-bit). It runs on Windows, not in WSL, because it needs a library WSL does not have. Only segments 1 and 2 are used from this step in the mixed setup, but all four are produced so you can experiment. `may has wrong signed offset` warnings are harmless noise.

In [ ]:
# 1) one-time x64 venv with the quantizer's Python dependencies
QENV = WORK / "qenv"
if not (QENV / "Scripts/python.exe").exists():
    subprocess.run([str(QPY), "-m", "venv", str(QENV)], check=True)
    pipenv = dict(os.environ)
    ns = Path(r"C:\ProgramData\Netskope\stagent\download")
    if (ns / "nscacert.pem").exists():                     # corporate TLS interception: build a CA bundle for pip
        bundle = WORK / "ns_bundle.pem"
        bundle.write_text(b"".join((ns / n).read_bytes() for n in ["nscacert.pem", "nstenantcert.pem"] if (ns / n).exists()).decode())
        pipenv["PIP_CERT"] = str(bundle)
    subprocess.run([str(QENV / "Scripts/python.exe"), "-m", "pip", "install", "-q", "numpy==1.26.4", "pyyaml==6.0.3", "packaging==24.0", "onnx==1.16.1", "protobuf"], check=True, env=pipenv)

# 2) quantize each segment (skipped if the output exists)
QOUT = Q45 / "quant"; QOUT.mkdir(exist_ok=True)
qenv = dict(os.environ, PYTHONPATH=str(QAIRT / "lib/python"), PATH=str(QAIRT / "lib/x86_64-windows-msvc") + os.pathsep + os.environ["PATH"])
for s in SEGS:
    out = QOUT / f"{s}.dlc"
    if out.exists() and out.stat().st_size > 0: print(s, "already quantized:", round(out.stat().st_size / 2**20), "MB"); continue
    t0 = time.time()
    r = subprocess.run([str(QENV / "Scripts/python.exe"), str(QAIRT / "bin/x86_64-windows-msvc/qairt-quantizer"),
                        "--input_dlc", str(FP32 / f"{s}.dlc"), "--output_dlc", str(out), "--input_list", str(CAL / f"{s}_list_win.txt"),
                        "--weights_bitwidth", str(W_BITS), "--act_bitwidth", str(A_BITS), "--bias_bitwidth", "32", "--float_bitwidth", "16",
                        "--preserve_io_datatype", "--use_per_channel_quantization"],
                       capture_output=True, text=True, env=qenv, cwd=str(Q45))
    (Q45 / f"{s}.quant.log").write_text(r.stdout + r.stderr)
    if r.returncode or not out.exists(): raise RuntimeError(f"{s}: quantizer failed rc={r.returncode}; see {Q45 / (s + '.quant.log')}\n" + (r.stdout + r.stderr)[-800:])
    print(f"{s}: {out.stat().st_size / 2**20:.0f} MB in {time.time() - t0:.0f}s (fp32 source {(FP32 / (s + '.dlc')).stat().st_size / 2**20:.0f} MB)")

## 11. Push the int8 graphs, build their context binaries, and gate the mix

Only segments 1 and 2 need int8 context binaries for the mixed setup, but all four are built so the folder layout is complete. Then the **mixed gate**: segments 0 and 3 from the fp16 folder, segments 1 and 2 from the int8 folder, compared with the fp32 reference. At 95 s this scored 0.99998; here it has to prove itself again for the new length.

In [ ]:
for s in SEGS: push(QOUT / f"{s}.dlc", f"{BOARD}/{SEGQ}")
build_contexts(SEGQ, CTXQ)

In [ ]:
cos_mix, err_mix, gout = run_gate(MIX, "mix", CASE)
print(gout)
print(f"mixed ({MIX}) vs fp32 PyTorch: cosine {cos_mix:.5f}, max abs err {err_mix:.3f}   (fp16 short graphs: {cos16:.5f})")
if cos_mix >= 0.999: print("-> as good as fp16: generate a song and listen")
elif cos_mix >= 0.99: print("-> usable, check by ear")
else: print("-> too lossy: do NOT generate a song")

## 12. Song: 45 seconds, mixed int8

Runs `infer_iq8.py` with `DR_FRAMES` set so DiffRhythm builds a 960-frame song. Edit `STYLE_PROMPT` and `LYRICS_45` in the configuration cell to change it. Lyrics: one line per `[mm:ss.xx]` timestamp, in order, all before 44 s, no empty lines, and **one extra line at the end that is not sung**. The cell prints the wall-clock time and the NPU time, which are the numbers to compare with the 95 s song (about 10 to 12 minutes in total, of which about 195 s on the NPU).

In [ ]:
assert "cos_mix" not in globals() or cos_mix >= 0.99, "mixed gate failed; not generating a song"   # (skipped when you run only sections 1 and 12)
def make_song(ctx, label, steps=None):
    """Generate the short song with the given context folders; return (wav path, wall minutes, log text)."""
    steps = steps or STEPS
    tag = time.strftime("%Y%m%d_%H%M") + f"_{int(SONG_SECONDS)}s_{steps}steps_{label}"
    run = f"{WSL_ROOT}/runs/iq8_{tag}"
    wsl(ENV + f"mkdir -p {run}; cat > {run}/lyrics.lrc <<'EOF'\n{LYRICS_45}EOF\ncp $S/infer_iq8.py $R/infer_iq8.py\n")
    t0 = time.time()
    _, lg = wsl(ENV + f"""
export PHONEMIZER_ESPEAK_LIBRARY=$($PY -c "import espeakng_loader as e;print(e.get_library_path())")
export ESPEAK_DATA_PATH=$($PY -c "import espeakng_loader as e;print(e.get_data_path())")
export OMP_NUM_THREADS=$(nproc)
export DR_STEPS={steps} DR_FRAMES={FRAMES} DR_CTX_DIR={ctx} IQ8_HOST={shlex.quote(IQ8_HOST)} IQ8_HOSTKEY={shlex.quote(IQ8_HOSTKEY)}
export IQ8_PW={shlex.quote(IQ8_PW)}
cd $R/src
$PY $R/infer_iq8.py --lrc-path {run}/lyrics.lrc --ref-prompt "{STYLE_PROMPT}" --audio-length 95 --output-dir {run} --chunked 2>&1 | grep -E "iq8[]] (short|sampler|total|pass failed)|Error|error|Traceback" | tail -40
""", timeout=4 * 3600)
    mins = (time.time() - t0) / 60
    dst = OUT / f"diffrhythm_iq8_npu_{tag}.wav"
    wsl(f"cp {run}/output.wav {to_wsl_path(OUT)}/{dst.name}")
    return dst, mins, lg

def stats(path):
    with wave.open(str(path)) as w:
        n, ch, sr = w.getnframes(), w.getnchannels(), w.getframerate(); a = array.array("h", w.readframes(n))
    return (f"{n/sr:6.1f} s | rms {math.sqrt(sum(x*x for x in a)/len(a))/32768:.3f} | peak {max(abs(x) for x in a)/32768:.3f}"
            f" | clipped {sum(1 for x in a if abs(x) >= 32767)/len(a):.4%}")

song_mix, mins_mix, log_mix = make_song(MIX, f"mixed_{SCHEME}")
print(f"wall {mins_mix:.1f} min ->", song_mix.name)
print("mixed int8, short:", stats(song_mix))
for line in log_mix.splitlines():
    if "total passes" in line: print(line)

## 13. (Optional) The same short song with all-fp16 graphs

Makes the same song (same prompt, lyrics and steps) with the fp16 short graphs, so you can compare **speed** (wall clock and NPU time) and **sound** (listen to both) against the mixed int8 song above. About the same wall time as section 12 plus a little more NPU time.

In [ ]:
song_fp16, mins_fp16, log_fp16 = make_song(CTX16, "fp16")
print(f"wall {mins_fp16:.1f} min ->", song_fp16.name)
print("fp16, short       :", stats(song_fp16))
print("mixed int8, short :", stats(song_mix))
for tag, lg in (("fp16 ", log_fp16), ("mixed", log_mix)):
    for line in lg.splitlines():
        if "total passes" in line: print(tag, line)
print(f"\nwall clock: mixed int8 {mins_mix:.1f} min, fp16 {mins_fp16:.1f} min.  For reference, the 95 s song took about 10 to 12 min (mixed int8).")

## 14. Play the song on the IQ8

Pushes the wav to the board and plays it through the board's own audio output (PipeWire). It needs a speaker or headphones on the board's audio jack; without one the player still runs but nothing is audible. The song is also in `diffrhythm_outputs/` on the PC.

In [ ]:
PLAY_WAV = song_mix if "song_mix" in globals() else sorted(OUT.glob("diffrhythm_iq8_npu_*_s_*.wav"))[-1]
PLAY_SECONDS = 60
push(PLAY_WAV, f"{BOARD}/play")
_ = ssh(f"""cd {BOARD}/play; pkill -x pw-play; rm -f play.log
setsid nohup timeout {PLAY_SECONDS} pw-play {PLAY_WAV.name} > play.log 2>&1 < /dev/null &
sleep 4; echo "--- log after 4 s:"; cat play.log; pgrep -a pw-play | cut -c1-80""")
print(f"Playing {PLAY_WAV.name} on the IQ8 (stops by itself after at most {PLAY_SECONDS} s).")

## Troubleshooting

- **Export fails or the chained check fails (section 3):** the export asserts that the four chunks equal the full model before writing anything. If it fails for 960 frames, try another length (`DR_FRAMES` in the environment, a multiple of 32 up to 2048) and delete `seg_45s` in WSL first.
- **Fp16 gate cosine below 0.99 (section 6):** do not continue. The graph and the test case disagree; delete `ctx_45s` on the board and `seg_45s` in WSL and rebuild.
- **Mixed gate fails (section 11):** the int8 calibration did not transfer to the new length. Add more prompts to `CALIB_PROMPTS`, delete `calib_dump_45s`, `q45_w8a16` and `ctx_45s_w8a16`, and rerun sections 7 to 11. Percentile calibration was tried on the 95 s model and made it worse; do not use it.
- **`stale dit_server on the board`:** a hung NPU process. Reboot the board, then re-run.
- **Quantizer exits with code 1 and no message in WSL:** run it on Windows as in section 10.
- **`pip` certificate error creating `qenv`:** a TLS-inspecting proxy; section 10 builds a CA bundle from the Netskope certificates when they exist, otherwise point `PIP_CERT` at your proxy's CA bundle.
- **Board resets during context generation or the gate:** the known memory fault (see `DIFFRHYTHM_IQ8.md`); the loops wait and retry.
- **The song cuts off or the lyrics are missing:** keep all lyric lines before 44 s, remember the last line is not sung, and never leave an empty line.